<a href="https://colab.research.google.com/github/saithrisank12/RAG_working/blob/main/RAG_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***📘 SECTION 1 — What is RAG and Why Do We Use It?***



RAG = Retrieval Augmented Generation
RAG is a technique where an LLM retrieves relevant information from external documents or a knowledge source and uses that information to generate an answer.

# ***🤔 Why do we need RAG?***
An LLM already has knowledge from its training data, but it may not know:
- Your private documents
- Your college notes
- Company policies
- Your project reports
- Newly added information
- Information stored in your database

# SECTION 2 — Load Our Document

RAG needs some knowledge to search.
So first, we need to give our RAG system a document.
For learning, instead of starting with a complicated PDF, we'll create a small text document ourselves.
Our document will contain information about an AI & ML department.

In [51]:
# Our knowledge document

document = """
SNIST AI & ML Department

The department offers a B.Tech program in Artificial Intelligence
and Machine Learning.

Students learn Machine Learning, Deep Learning,
Natural Language Processing and Computer Vision.

The department encourages students to participate
in research projects and technical competitions.
"""

print(document)


SNIST AI & ML Department

The department offers a B.Tech program in Artificial Intelligence
and Machine Learning.

Students learn Machine Learning, Deep Learning,
Natural Language Processing and Computer Vision.

The department encourages students to participate
in research projects and technical competitions.



# SECTION 3 — Clean & Prepare the Text

Our document may contain:
- Extra spaces
- Empty lines
- Unnecessary formatting
- Broken text
Before we divide the document into chunks, we want to make the text clean and consistent.

In [52]:
# Clean extra spaces and empty lines

cleaned_text = " ".join(document.split())

print(cleaned_text)

SNIST AI & ML Department The department offers a B.Tech program in Artificial Intelligence and Machine Learning. Students learn Machine Learning, Deep Learning, Natural Language Processing and Computer Vision. The department encourages students to participate in research projects and technical competitions.


## We have the document → we clean it → now we need to break it up → because later we need to find only the relevant information → therefore we create chunks → then we need a way to compare their meaning → therefore embeddings...

#  Section 4

***What is chunking?***


Chunking means breaking a large document into smaller pieces while keeping each piece as a complete and meaningful idea.

Instead of cutting the document after a fixed number of words, we split it using paragraphs so phrases such as “Natural Language Processing” stay together.

These chunks will later be converted into embeddings and stored in ChromaDB, so good chunking helps the retrieval system find the right information more accurately.


In [53]:
# Split the document into meaningful paragraphs

chunks = [
    paragraph.strip().replace("\n", " ")
    for paragraph in document.strip().split("\n\n")
    if paragraph.strip()
]

# Display the chunks
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk)


--- Chunk 1 ---
SNIST AI & ML Department

--- Chunk 2 ---
The department offers a B.Tech program in Artificial Intelligence and Machine Learning.

--- Chunk 3 ---
Students learn Machine Learning, Deep Learning, Natural Language Processing and Computer Vision.

--- Chunk 4 ---
The department encourages students to participate in research projects and technical competitions.


# 🔢 SECTION 5 — Embeddings

We now have our document divided into smaller chunks.

The next problem is:

> **How can a computer understand which chunk is related to a user's question?**

For example, our document may contain:

> "Students learn Machine Learning and Deep Learning."

Later, a user might ask:

> "What subjects do students study?"

The words are different, but the **meaning is related**.

To help the computer compare this meaning, we convert each text chunk into numbers. This numerical representation is called an **embedding**.

## 🧠 What is an Embedding?

An embedding is a **vector of numbers that represents the meaning of a piece of text**.

Example

"Students learn Machine Learning."

              ↓
       Embedding Model (all-MiniLM-L6-v2)

              ↓
[0.21, -0.45, 0.73, 0.18, ...]

In [54]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

# Load a small embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings for our chunks
chunk_embeddings = embedding_model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 4
Embedding shape: (4, 384)


# we got output as :
Number of chunks: 2

Embedding shape: (2, 384)


2    → number of chunks

384  → numbers representing each chunk

# Let's actually see one embedding

In [55]:
print("First chunk:")
print(chunks[0])

print("\nEmbedding of the first chunk:")
print(chunk_embeddings[0])

First chunk:
SNIST AI & ML Department

Embedding of the first chunk:
[-1.52843678e-02 -1.03361219e-01 -3.84954512e-02 -4.17768247e-02
  1.65898819e-02  6.75385743e-02 -5.57173528e-02  2.00769939e-02
 -1.80257503e-02  4.72214408e-02  1.04746446e-02  1.24852434e-02
  3.09053622e-02 -8.36897120e-02 -7.97104463e-02  7.97885954e-02
  4.48273383e-02  3.95321008e-03 -7.42491186e-02 -8.75775069e-02
 -1.45290475e-02  7.08104111e-03 -4.34556901e-02  4.81697433e-02
 -6.28073812e-02  3.32413986e-02 -3.83500867e-02 -6.46869540e-02
 -1.18436497e-02  1.25046959e-03  8.27632658e-03 -1.18436307e-01
  9.03468356e-02  3.33717835e-05 -2.00117696e-02  7.69348964e-02
  3.13200541e-02  1.79321095e-02  8.65014121e-02  2.03855485e-02
  3.80689278e-02 -6.80700466e-02  3.11797601e-04 -9.22649577e-02
  4.23509590e-02  8.28884467e-02 -6.06392231e-03 -4.47296537e-02
 -9.10541937e-02  2.05071438e-02  1.41518274e-02 -8.31904858e-02
  7.83856027e-03  3.47290970e-02 -4.83566448e-02 -4.76459712e-02
 -2.45716097e-03 -1.0

# 🗄️ SECTION 6 — Vector Database

## 📖 Explanation

So far, we have taken our document, cleaned the text, divided it into chunks, and converted each chunk into a numerical representation called an embedding.

 For example, a text chunk such as "Students learn Machine Learning and Deep Learning" is converted into a vector containing 384 numbers when we use the `all-MiniLM-L6-v2` embedding model.

 Now we need to store these vectors somewhere so that we can search them efficiently later. This is where a **Vector Database** is used.

  A vector database is designed to store embeddings and the original text associated with them. We will use **ChromaDB** for this project because it is simple, can run locally in Google Colab, and does not require us to create an account. We will store each chunk together with its embedding.
  
   The embedding will later help us find information by meaning, while the original chunk will give us the actual text that we can provide to the LLM.
   
    In simple terms, we are now doing: Text Chunk → Embedding → Store both in ChromaDB . Once this step is complete, our document knowledge is stored and ready to be searched when a user asks a question.

In [56]:
# Install ChromaDB
!pip install -q chromadb

import chromadb

# Create ChromaDB client
client = chromadb.Client()

# Create a collection
collection = client.get_or_create_collection("snist_ai_ml")

# Store chunks and their embeddings
for i, (chunk, embedding) in enumerate(zip(chunks, chunk_embeddings)):
    collection.add(
        ids=[str(i)],
        documents=[chunk],
        embeddings=[embedding.tolist()]
    )

print("Number of stored chunks:", collection.count())

Number of stored chunks: 4


# SECTION 7 — Take a User Query


Now that our document chunks and their embeddings are stored in ChromaDB, the next step is to simulate what happens when a real user asks a question. The user does not give us an embedding directly; they simply type a normal question in natural language.

For example, the user may ask ***“What subjects do students learn?”***.

We will store this question in a variable called query. In the next section, we will convert this query into an embedding using the same embedding model we used for our document chunks, so that the query can be compared with the stored vectors in ChromaDB.

In [57]:
# User's question

query = "What subjects do students learn?"

print("User query:", query)

User query: What subjects do students learn?


# SECTION 8 — Query Embedding

The user has given us a normal text question, but ChromaDB searches using numerical vectors, so we need to convert the question into an embedding first. We use the same all-MiniLM-L6-v2 model that we used for our document chunks, because both the document chunks and the query must be represented in the same vector space to compare their meaning.

 For example, the question "What subjects do students learn?" will be converted into a vector containing 384 numbers.

 These numbers together represent the semantic meaning of the entire question, not one number for each word.

 We will then use this query embedding to search the embeddings already stored in ChromaDB and find the chunks that are most relevant to the question.

In [58]:
# Convert the user's query into an embedding

query_embedding = embedding_model.encode(query)

print("Query:", query)
print("Embedding shape:", query_embedding.shape)

Query: What subjects do students learn?
Embedding shape: (384,)


# SECTION 9 — Semantic Search & Retrieve Relevant Chunks

Now we have the query embedding and the document embeddings stored in ChromaDB, so we can finally search for information that is most similar in meaning to the user's question.

Instead of looking only for exact matching words, ChromaDB compares the query vector with the stored vectors and calculates their similarity.

For example, the query “What subjects do students learn?” may match a chunk containing “Students learn Machine Learning, Deep Learning, Natural Language Processing and Computer Vision”,

even though the wording is not exactly the same. We will ask ChromaDB to return the most relevant result using our query embedding.

The n_results=2 below means we want the two most similar chunks.

We immediately extract the actual text from the search results and store it in relevant_chunks. This gives us the information that will be used as context for the LLM in the next step.

In [59]:
# Search ChromaDB using the query embedding

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2
)

# Extract the retrieved text chunks
relevant_chunks = results["documents"][0]

# Display the retrieved chunks
print("Relevant chunks:\n")

for i, chunk in enumerate(relevant_chunks):
    print(f"Chunk {i + 1}:")
    print(chunk)

Relevant chunks:

Chunk 1:
Students learn Machine Learning, Deep Learning, Natural Language Processing and Computer Vision.
Chunk 2:
Processing and Computer Vision. The department encourages students to participate in research projects and technical competitions.


# SECTION 10 — Give Context to LLM and Generate Answer

We have now retrieved the chunks that are relevant to the user's question. The next step is to give this retrieved information to an LLM as context.

 We combine the retrieved chunks into one piece of text and create a prompt containing the context and the original question. The LLM will read this information and generate the final answer.

 This is the Generation part of RAG: retrieval found the information, augmentation adds it to the prompt, and generation produces the answer.

 We are not training or fine-tuning the LLM here; we are simply providing additional information to the model while answering the question.


In [60]:
# Combine retrieved chunks into context
context = "\n".join(relevant_chunks)

# Create the prompt
prompt = f"""
Answer the question using only the information provided in the context.

Context:
{context}

Question:
{query}

Answer:
"""

print(prompt)


Answer the question using only the information provided in the context.

Context:
Students learn Machine Learning, Deep Learning, Natural Language Processing and Computer Vision.
Processing and Computer Vision. The department encourages students to participate in research projects and technical competitions.

Question:
What subjects do students learn?

Answer:



# SECTION 11 — Generate the Answer using Groq

Up to this point, our RAG system has retrieved the information that is relevant to the user's question and combined those retrieved chunks into a context.

However, ChromaDB is only responsible for finding information; it does not write the final natural-language answer. For that, we need an LLM.

 We will use Groq to access an LLM through its API. Groq is a cloud inference platform,
We send our RAG prompt, which contains the retrieved context and the user's query, to the Groq model.

 The model reads that information and generates the final answer.

  This completes our RAG process:
  
  Retrieval finds the relevant information, Augmentation adds that information to the prompt, and Generation produces the answer.

In [61]:
!pip install -q groq

from groq import Groq

# Your Groq API key
api_key = "gsk_********************************".strip()

# Create Groq client
client = Groq(api_key=api_key)

# Send the RAG prompt to Groq
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

# Final answer
print("Final Answer:")
print(response.choices[0].message.content)

Final Answer:
Students learn **Machine Learning, Deep Learning, Natural Language Processing, and Computer Vision**.


***For testing, you can use a Groq API key, but I can’t provide or expose my own private API key. You should use your own newly generated Groq key.***